# Week 44, Day 4: How Python's dict works

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w044d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
d = {"b": 1, "a": 2, "c": 3}
del d["b"]
d["b"] = 4
d["a"] = 5
print(list(d), d)
print({1: "int", True: "bool", 1.0: "float"})

In [ ]:
def probes(h, size, count):
    mask = size - 1
    perturb = h
    i = h & mask
    order = [i]
    for _ in range(count - 1):
        perturb >>= 5
        i = (5 * i + perturb + 1) & mask
        order.append(i)
    return order

print(probes(12345, 8, 8))
print(probes(12345 + 8 * 1000, 8, 8))        # same low bits, so the same first slot, then different paths
print(probes(0, 8, 8))                       # perturb is 0: the plain 5i + 1 sequence visits every slot

In [ ]:
class Noisy:
    def __init__(self, name, h):
        self.name, self.h = name, h
    def __hash__(self):
        return self.h
    def __eq__(self, other):
        print(f"  comparing {self.name} with {other.name}")
        return self.name == other.name

a, b, c = Noisy("a", 1), Noisy("b", 9), Noisy("c", 1)
d = {a: 1, b: 2}
print("lookup a itself:"); d[a]
print("lookup c (same hash as a, different key):"); print(c in d)
print("lookup b's twin (same hash, equal name):"); print(Noisy("b", 9) in d)

In [ ]:
import sys

d = {}
last = sys.getsizeof(d)
for i in range(50):
    d[i] = i
    size = sys.getsizeof(d)
    if size != last:
        print(f"after {len(d):>2} keys: {size} bytes")
        last = size

> **Key idea.** CPython's dict is a compact, open-addressed hash table: a dense list of `(hash, key, value)` entries in insertion order, and a sparse power-of-two index table of small numbers pointing into it (Raymond Hettinger's design, in CPython since 3.6). Iterating the entries gives insertion order, which Python 3.7 guarantees; updating a value keeps a key's place, deleting and re-adding moves it to the end. Probing starts at `hash & mask` and follows `i = 5i + perturb + 1`, shifting `perturb` right by 5 each step to mix in the high bits. Keys are compared by identity, then by stored hash, then by `==`. Tables grow before two thirds full, O(1) amortised.

### Exercise: Order rules

Write `final_order(ops)`: run a list of operations on an empty dict and return its keys in order. Each operation is `("set", key, value)`, `("del", key)` or `("update", other_dict)`. Use a real dict (the point is to learn its rules). Then set the four answers below, which describe what the rules do.

In [ ]:
def final_order(ops):
    ___

p = ___   # final_order([("set", "a", 1), ("set", "b", 2), ("set", "a", 3)])
q = ___   # final_order([("set", "a", 1), ("set", "b", 2), ("del", "a"), ("set", "a", 3)])
r = ___   # final_order([("set", "x", 1), ("update", {"y": 2, "x": 3, "z": 4})])
s = ___   # list({True: "a", 1: "b", 1.0: "c"}.items())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_order():\n    \"dict order\"\n    assert final_order([]) == [] and final_order([(\"set\", 3, 0), (\"set\", 1, 0), (\"set\", 2, 0)]) == [3, 1, 2]\n    assert p == [\"a\", \"b\"] and q == [\"b\", \"a\"] and r == [\"x\", \"y\", \"z\"] and s == [(True, \"c\")]")

### Exercise: CPython's probe sequence

Write `probe_slots(h, size, n)`: the first `n` slots that CPython's probe sequence visits for a hash `h` (a whole number of 0 or more) in an index table of `size` slots (a power of two): start at `h & mask`; then, each step, shift `perturb` (starting as `h`) right by 5 and move to `(5 * i + perturb + 1) & mask`. Then write `covers_all(size)`: true if the plain sequence (`h = 0`) visits every slot within `size` steps.

In [ ]:
def probe_slots(h, size, n):
    ___

def covers_all(size):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_probe():\n    \"probing\"\n    assert probe_slots(0, 8, 8) == [0, 1, 6, 7, 4, 5, 2, 3] and probe_slots(5, 8, 1) == [5]\n    assert probe_slots(12345, 8, 4) == [1, 7, 0, 1]\n    assert covers_all(8) and covers_all(16) and covers_all(1024)\n    assert probe_slots(1, 8, 3) != probe_slots(1 + 32, 8, 3), \"high bits change the path\"")

### Exercise: A compact, ordered table

Build `CompactDict`, a model of CPython's layout: `_indices`, a list of `size` slots (start with 8) holding `None` or a position in `_entries`; and `_entries`, a list of `[hash, key, value]` in insertion order. Probe CPython's way, as in `probe_slots`. Write `__setitem__` (updating an existing key keeps its place), `__getitem__` (`KeyError` if missing), `__len__`, `__iter__` (keys in insertion order) and `items()`. When the number of entries would go over two thirds of `size`, rebuild with `size` doubled, reinserting the entries in order. Compare keys by identity, then by stored hash, then by `==`.

In [ ]:
class CompactDict:
    def __init__(self):
        self._indices = [None] * 8
        self._entries = []

    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_basic():\n    \"a compact dict\"\n    d = CompactDict()\n    for k, v in [(\"b\", 1), (\"a\", 2), (\"c\", 3), (\"a\", 9)]:\n        d[k] = v\n    assert list(d) == [\"b\", \"a\", \"c\"] and d[\"a\"] == 9 and len(d) == 3 and d.items() == [(\"b\", 1), (\"a\", 9), (\"c\", 3)]\n    try:\n        d[\"z\"]\n        assert False\n    except KeyError:\n        pass\n    assert all(isinstance(x, int) or x is None for x in d._indices) and len(d._entries) == 3\ndef test_growth():\n    \"growing keeps order\"\n    d = CompactDict()\n    keys = random.Random(1).sample(range(10 ** 6), 500)\n    for i, k in enumerate(keys):\n        d[k] = i\n    assert list(d) == keys and all(d[k] == i for i, k in enumerate(keys))\n    assert len(d._indices) >= 750 and len(d._indices) & (len(d._indices) - 1) == 0, \"a power of two, at most two thirds full\"\n    d[1] = \"one\"\n    d[1.0] = \"float\"\n    assert d[True] == \"float\" and list(d)[-1] == 1 and len(d) == 501")

### Exercise: Counting comparisons

`Key` (in the starter) counts every call of its `__eq__` in `Key.comparisons`. Set the four answers to the number of `==` comparisons each lookup makes, by experiment or by reasoning from today's three-step check. Then write `same_object_free(d, key)`: look `key` up in `d` and return `True` if that lookup made **no** comparisons.

In [ ]:
class Key:
    comparisons = 0
    def __init__(self, name, h):
        self.name, self.h = name, h
    def __hash__(self):
        return self.h
    def __eq__(self, other):
        Key.comparisons += 1
        return isinstance(other, Key) and self.name == other.name

a, b = Key("a", 1), Key("b", 2)
d = {a: "A", b: "B"}

def same_object_free(d, key):
    ___

p = ___   # comparisons for d[a]: the very same object
q = ___   # comparisons for d[Key("a", 1)]: an equal key, a different object
r = ___   # comparisons for Key("z", 3) in d: a hash nothing else has
s = ___   # comparisons for Key("c", 1) in d: same hash as a, but not equal

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_counts():\n    \"cheap comparisons\"\n    def count(f):\n        Key.comparisons = 0\n        f()\n        return Key.comparisons\n    assert p == count(lambda: d[a]) == 0\n    assert q == count(lambda: d[Key(\"a\", 1)]) == 1\n    assert r == count(lambda: Key(\"z\", 3) in d) == 0\n    assert s == count(lambda: Key(\"c\", 1) in d) >= 1\n    assert same_object_free(d, a) is True and same_object_free(d, Key(\"a\", 1)) is False")

### Exercise: Using the order

Dicts keep insertion order, so they can do jobs that once needed extra work. Write three one-pass functions, each using a dict: `unique(items)` (the first appearance of each item, in order); `last_seen(events)` (for a list of `(name, time)` pairs, a dict from each name to its latest time, with the names ordered by when they were **last** seen, earliest first: delete and re-add a name to move it to the end); and `first_repeat(items)` (the first item that appears for a second time, or `None`).

In [ ]:
def unique(items):
    ___

def last_seen(events):
    ___

def first_repeat(items):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_order_tricks():\n    \"using the order\"\n    assert unique([3, 1, 3, 2, 1, 4]) == [3, 1, 2, 4] and unique([]) == []\n    got = last_seen([(\"ada\", 1), (\"bo\", 2), (\"ada\", 3), (\"chi\", 4), (\"bo\", 5)])\n    assert got == {\"ada\": 3, \"chi\": 4, \"bo\": 5} and list(got) == [\"ada\", \"chi\", \"bo\"]\n    assert first_repeat(\"abcbad\") == \"b\" and first_repeat([1, 2, 3]) is None")

### Bug hunt: Looping in hash order

This model iterates over the **index table**, which is in hash order, instead of over the entries, so the keys come out in slot order, not insertion order: it prints `[0, 1, 2, 4, 9]`. Iterate over the entries, so it prints `[9, 1, 4, 0, 2]`.

In [ ]:
indices = [None] * 16
entries = []

def add(key, value):
    i = hash(key) & 15
    while indices[i] is not None:
        i = (i + 1) & 15
    indices[i] = len(entries)
    entries.append((key, value))

def keys():
    return [entries[p][0] for p in indices if p is not None]

for k in [9, 1, 4, 0, 2]:
    add(k, str(k))
print(keys())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"insertion order\"\n    expect_output(\"[9, 1, 4, 0, 2]\")")

### Bug hunt: The mask that isn't

For a table of 8 slots, the mask must be `7` (binary `111`), so `h & mask` is from 0 to 7. This uses the size itself (`8`, binary `1000`), so `h & 8` is only ever 0 or 8, and slot 8 doesn't exist: it crashes with `IndexError`. Fix the mask so this prints `[None, 9, 2, None, None, None, None, 15]`.

In [ ]:
size = 8
mask = size
slots = [None] * size
for key in [9, 2, 15]:
    i = key & mask
    while slots[i] is not None:
        i = (i + 1) & mask
    slots[i] = key
print(slots)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"size minus one\"\n    expect_output(\"[None, 9, 2, None, None, None, None, 15]\")")

### Bug hunt: Equal hashes aren't equal keys

This lookup decides it has found the key as soon as the stored **hash** matches, without comparing the keys themselves, so a different key that happens to share a hash gets the wrong value: this prints `apple`. Compare the keys too, so it prints `None`.

In [ ]:
entries = []

def add(key, value):
    entries.append((hash(key), key, value))

def get(key):
    h = hash(key)
    for eh, ek, v in entries:
        if eh == h:
            return v
    return None

class Code:
    def __init__(self, s):
        self.s = s
    def __hash__(self):
        return len(self.s)
    def __eq__(self, other):
        return isinstance(other, Code) and self.s == other.s

add(Code("AB1"), "apple")
print(get(Code("XY9")))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"compare the keys\"\n    expect_output(\"None\")\n    assert get(Code(\"AB1\")) == \"apple\"")

### Puzzle: CPython's probe loop

Arrange the lines into a function that finds the index-table slot for a hash, CPython's way. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
d = {"x": 1, "y": 2}
d["x"] = 10
a = list(d)
del d["x"]
d["x"] = 11
b = list(d)
c = {**{"a": 1, "b": 2}, **{"b": 3, "c": 4}}

p = ___   # a
q = ___   # b
r = ___   # list(c.items())
s = ___   # 12345 & 7
t = ___   # the fraction of the index table a CPython dict lets be used before growing

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [\"x\", \"y\"] and q == [\"y\", \"x\"] and r == [(\"a\", 1), (\"b\", 3), (\"c\", 4)], \"p, q and r\"\n    assert s == 1 and t in (\"2/3\", 2 / 3), \"s and t (write t as the string '2/3' or the number)\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a complete compact dict

Extend the compact model into `OrderedTable` with deletion. Keep CPython's design: `_indices` (power-of-two size, starting at 8) holding `None` (empty), `DELETED` (a deleted slot, like Day 3's tombstone) or an entry position; and `_entries`, where a deleted entry becomes `None` (so the others keep their positions and order). Give it `__setitem__`, `__getitem__`, `__delitem__` (`KeyError` if missing), `__contains__`, `__len__` (live entries), `__iter__` (insertion order, skipping deleted entries), `items()`, and `popitem()` (remove and return the **most recently added** live pair, as `dict.popitem` does; `KeyError` if empty). When `len(_entries)` (including deleted) would go over two thirds of the index size, rebuild: squeeze out deleted entries, and choose the smallest power-of-two size (at least 8) that keeps the live entries under a third full. Use CPython's probe sequence.

In [ ]:
DELETED = object()

class OrderedTable:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_basic():\n    \"basics\"\n    t = OrderedTable()\n    for k in \"dbca\":\n        t[k] = k.upper()\n    del t[\"b\"]\n    t[\"b\"] = \"again\"\n    assert list(t) == [\"d\", \"c\", \"a\", \"b\"] and len(t) == 4 and t[\"b\"] == \"again\" and \"b\" in t and \"z\" not in t\n    assert t.popitem() == (\"b\", \"again\") and t.popitem() == (\"a\", \"A\") and list(t) == [\"d\", \"c\"]\n    for bad in [lambda: t[\"b\"], lambda: t.__delitem__(\"b\"), lambda: OrderedTable().popitem()]:\n        try:\n            bad()\n            assert False\n        except KeyError:\n            pass\ndef test_random():\n    \"against a dict\"\n    rng = random.Random(3)\n    t, ref = OrderedTable(), {}\n    for _ in range(6000):\n        k = rng.randint(0, 200)\n        r = rng.random()\n        if r < 0.55:\n            t[k] = ref[k] = rng.randint(0, 9)\n        elif r < 0.85 and k in ref:\n            del t[k], ref[k]\n        elif r < 0.9 and ref:\n            assert t.popitem() == ref.popitem()\n        assert (k in t) == (k in ref) and len(t) == len(ref)\n    assert list(t) == list(ref) and t.items() == list(ref.items())\n    size = len(t._indices)\n    assert size & (size - 1) == 0 and len(t._entries) * 3 <= size * 2")

## Warm-up

Two surprises. A dict lists its keys in the order they were **first added**, and updating a value doesn't move a key, but deleting and re-adding it does. And `1`, `True` and `1.0` are equal and hash alike (Day 1), so they're **one** key: the first spelling (`1`) is kept, and each later assignment just replaces the value.

## Two tables, not one

Until Python 3.6, a dict was one big open-addressed table of `(hash, key, value)` entries, about a third of them empty at any time, each empty slot as big as a full one. In 2012, Raymond Hettinger proposed splitting it in two (the design was built into CPython 3.6 by INADA Naoki):

- a dense list of **entries**, `(hash, key, value)`, in the order keys were added, with no gaps except for deletions;
- a sparse **index table**, a power-of-two number of slots, each holding just a small whole number: the position of an entry, or a marker for empty or deleted.

```text
d = {"timmy": "red", "barry": "green", "guido": "blue"}

indices:  [None, 1, None, None, None, 0, None, 2]        (8 slots, small numbers)

entries:  0: (hash("timmy"), "timmy", "red")
          1: (hash("barry"), "barry", "green")             (in insertion order)
          2: (hash("guido"), "guido", "blue")
```

Hashing and probing happen in the index table; the slots there hold only small numbers, so the empty ones cost very little. The entries hold the real data, packed tightly. A dict of a few keys uses a fraction of the memory the old layout needed, and looping over it walks the dense entries list, which is fast.

## Order for free

Because entries are appended in order, the dict **remembers insertion order** without any extra work: iterating over `entries` gives keys in the order they were added. CPython 3.6 did this as a side effect of the new layout; Python 3.7 made it a promise of the language, so you can rely on it.

What happens to order on each operation:

| Operation | Effect on order |
|---|---|
| add a new key | it goes to the end |
| update an existing key's value | it stays where it is |
| delete a key | its entry is marked empty; the others keep their order |
| delete, then add the same key again | it goes to the end, as a new key |
| the dict grows (resizes) | entries are copied in order, and the deleted gaps are squeezed out |

That's why `dict.fromkeys(items)` removes duplicates while keeping the first appearance of each (Week 10), and why `collections.OrderedDict` is now needed only for its extra methods, like `move_to_end` (Week 42's LRU cache).

## The probe sequence

The index table uses open addressing (Day 3), but not linear probing. CPython's probe sequence starts at `hash & mask` (the low bits of the hash, where `mask` is the table size minus 1), and then moves with a recurrence that gradually mixes in the **higher** bits of the hash:

```text
perturb = hash                          (as an unsigned number)
i = hash & mask
while slot i is taken by a different key:
    perturb >>= 5
    i = (5 * i + perturb + 1) & mask
```

The `5 * i + 1` part alone would visit every slot of a power-of-two table eventually; adding `perturb` makes keys whose hashes differ only in their high bits take different paths, instead of crowding together as they would with linear probing. After a few steps `perturb` becomes 0 and the plain recurrence takes over, so the search always ends. (The full explanation is in the comments of CPython's `Objects/dictobject.c`, which are well worth reading.)

Notice that ints hash to themselves, so a dict of the keys 0 to 7 needs no probing at all: each goes straight to its own slot.

## Comparing keys quickly

When a probe reaches a slot holding an entry, is it the key we want? CPython checks in the cheapest order:

1. **Is it the very same object?** (`entry_key is key`): free, and very common, because names and strings are often the same object.
2. **Do the stored hashes differ?** If so, the keys are certainly different (equal keys have equal hashes), and no `==` is needed. This is why entries store the hash: it also avoids recomputing hashes when the table grows.
3. Only then: **`entry_key == key`**.

Looking up `a` itself needed no comparison at all; `c` shared `a`'s hash, so `==` was called; and the twin of `b` was found with one comparison. Strings also **cache** their hash the first time it's computed, so using the same string as a key again and again costs nothing extra.

## Growing, and what it means for you

A dict starts with an index table of 8 slots and lets at most two thirds of them be used; when it runs out of room, it builds a bigger table (several times the number of entries, rounded up to a power of two) and reinserts every entry, in order. As always with doubling-style growth, that's O(1) amortised per insert.

(The exact numbers vary between Python versions and computers.) What all this means in everyday code:

- **Lookups, inserts and deletes are O(1) on average**, but only if keys hash well. Your own classes' `__hash__` should mix all the fields that `__eq__` compares (hashing a tuple of them does that).
- **Iteration follows insertion order**, and is guaranteed to.
- **Don't add or remove keys while looping over a dict**: it raises `RuntimeError: dictionary changed size during iteration`. Loop over `list(d)` instead.
- **A dict uses more memory than a list** of the same values, for the index table and the stored hashes. For millions of small records, a list of tuples, or the `array` module, can be much smaller.

## Practice

## Quiz

## Daily challenge

## Before you go

Read the long comment near the top of CPython's `Objects/dictobject.c` (search the web for it). You'll recognise almost every idea in it from this week. Tomorrow: hashing problems, a day of classic puzzles that a dictionary or set turns from O(n²) into O(n).